# Phishing Detection under LLM Distribution Shift

**Curso:** CC3094 - Security Data Science  
**Proyecto:** Robustez de modelos de detección de phishing entrenados con correos humanos frente a correos generados por modelos de lenguaje grandes
**Miembros:** Edwin Ortega - 22305 y Esteban Zambrano 22119

### Objetivo de esta fase
En esta etapa se selecciona y prepara el dataset a utilizar, se realiza un análisis exploratorio de datos (EDA) y se generan/seleccionan características relevantes para la posterior implementación del modelo.

### Enfoque del proyecto
Aunque el dataset incluye correos humanos y correos generados por LLM, el objetivo final del proyecto es evaluar la robustez de modelos entrenados con correos humanos cuando se enfrentan a correos generados por LLM, sin reentrenamiento.

In [1]:
import pandas as pd
import numpy as np
import re
from pathlib import Path
import matplotlib.pyplot as plt
from collections import Counter

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import chi2

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 200)

In [2]:
# Encontrar la carpeta data

def find_data_dir():
    candidates = [Path("data"), Path("../data")]
    for path in candidates:
        if path.exists():
            return path
    raise FileNotFoundError("No se encontró la carpeta 'data' ni en la raíz ni en el nivel superior.")

DATA_DIR = find_data_dir()
DATA_DIR

WindowsPath('../data')

In [3]:
# Función especial para leer el CSV

def read_llm_phishing_broken_csv(file_path):
    rows = []
    with open(file_path, "r", encoding="utf-8", errors="replace") as f:
        lines = f.readlines()

    # Saltar encabezado
    for line in lines[1:]:
        line = line.strip()
        if not line:
            continue
        
        # Separar por la última coma para recuperar text,label
        try:
            text, label = line.rsplit(",", 1)
            rows.append({"text": text.strip(), "label": int(label.strip())})
        except ValueError:
            # Si alguna línea no se puede parsear, se ignora
            continue

    return pd.DataFrame(rows)

In [4]:
# Cargar los 4 datasets

human_legit = pd.read_csv(DATA_DIR / "human-generated" / "legit.csv")
human_phishing = pd.read_csv(DATA_DIR / "human-generated" / "phishing.csv")
llm_legit = pd.read_csv(DATA_DIR / "llm-generated" / "legit.csv")
llm_phishing = read_llm_phishing_broken_csv(DATA_DIR / "llm-generated" / "phishing.csv")

datasets = {
    "human_legit": human_legit,
    "human_phishing": human_phishing,
    "llm_legit": llm_legit,
    "llm_phishing": llm_phishing
}

In [5]:
# Inspección inicial

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 60)
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    display(df.head(2))


human_legit
------------------------------------------------------------
Shape: (1000, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']


,sender,receiver,date,subject,body,urls,label
0,Jesus Miguel Recuenco Ezquerra <JMRECU@teleline.es>,handy board <handyboard@media.mit.edu>,2019-10-29 22:53:50,Starting IC with wizard,"Hi\r\n\r\n\t\tI am running the IR test program from Max Davies. To do this I need to start\r\nIC with thw wizard option. As I have a macintosh, there is a wizard option for\r\nthe mac version of I...",0,0
1,Trade Me <xfnbqb@trademe.co.nz>,user2.4@gvc.ceas-challenge.cc,2008-08-06 13:53:26,Trade Me -- A question on your auction: Auction 130939357 for Small\tbookshelf,\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\nSecurity Note: Trade Me will never ask you for your password via email\r\n\r\n\r\n\r\nThis is an automated email regarding listing reference: 13...,0,0



human_phishing
------------------------------------------------------------
Shape: (1000, 7)
Columns: ['sender', 'receiver', 'date', 'subject', 'body', 'urls', 'label']


,sender,receiver,date,subject,body,urls,label
0,MetaMask <info@sofamekar.com>,jose@monkey.org,2022-12-27 10:56:49,Your MetaMask wallet will be suspended,"Verify your MetaMask Wallet Our system has shown that your MetaMask wallet has not yet been verified, this verification can be done easily via the button below. Unverified accounts will be suspe...",1,1
1,MetaMask <info@sofamekar.com>,jose@monkey.org,2022-12-27 10:56:49,Your MetaMask wallet will be suspended,"Verify your MetaMask Wallet Our system has shown that your MetaMask wallet has not yet been verified, this verification can be done easily via the button below. Unverified accounts will be suspe...",1,1



llm_legit
------------------------------------------------------------
Shape: (1000, 2)
Columns: ['text', 'label']


,text,label
0,"Dear Michael, I hope this message finds you well. We are reaching out to you with an urgent request regarding your account verification. As part of our commitment to ensuring the security of our u...",1
1,"Dear Jennifer, We hope you're doing well. We're writing to remind you of the importance of verifying your account with us. Your account security is paramount to us, and completing this verificatio...",1



llm_phishing
------------------------------------------------------------
Shape: (1000, 2)
Columns: ['text', 'label']


,text,label
0,"Dear User, We have received reports indicating that your account has been flagged for suspicious activities. To ensure the safety of your account and prevent any unauthorized access, we require yo...",1
1,"Dear Sarah Thompson, I hope this email finds you in good health. My name is Jessica Wilson, and I'm a senior recruiter at Global Solutions. We recently came across your profile on a renowned job p...",1


In [6]:
# Verificación de nulos, duplicados y 

for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 60)
    print("Nulos:")
    print(df.isnull().sum())
    print("\nDuplicados:", df.duplicated().sum())
    if "label" in df.columns:
        print("\nDistribución de label:")
        print(df["label"].value_counts(dropna=False))


human_legit
------------------------------------------------------------
Nulos:
sender       0
receiver    16
date         0
subject      0
body         0
urls         0
label        0
dtype: int64

Duplicados: 0

Distribución de label:
label
0    1000
Name: count, dtype: int64

human_phishing
------------------------------------------------------------
Nulos:
sender       0
receiver    22
date         0
subject      2
body         0
urls         0
label        0
dtype: int64

Duplicados: 496

Distribución de label:
label
1    1000
Name: count, dtype: int64

llm_legit
------------------------------------------------------------
Nulos:
text     0
label    0
dtype: int64

Duplicados: 2

Distribución de label:
label
1    1000
Name: count, dtype: int64

llm_phishing
------------------------------------------------------------
Nulos:
text     0
label    0
dtype: int64

Duplicados: 0

Distribución de label:
label
1    1000
Name: count, dtype: int64


## Estandarización del dataset

Los archivos humanos y los generados por LLM no tienen exactamente la misma estructura:
- Los correos humanos incluyen columnas como `subject`, `body`, `urls`, `sender`, etc.
- Los correos LLM contienen principalmente una columna `text`.

Para unificar el análisis, se construirá un dataset común con las siguientes columnas principales:

- `text`: texto completo del correo
- `source_type`: `human` o `llm`
- `email_group`: `legit` o `phishing`
- `label_binary`: 0 para legítimo, 1 para phishing

Además, se conservarán metadatos originales cuando estén disponibles.

In [7]:
# Funciones para unificar formato

def combine_human_email_text(df):
    subject = df["subject"].fillna("").astype(str)
    body = df["body"].fillna("").astype(str)
    return (subject + " " + body).str.strip()

def standardize_human_df(df, email_group):
    out = df.copy()
    out["text"] = combine_human_email_text(out)
    out["source_type"] = "human"
    out["email_group"] = email_group
    out["label_binary"] = 0 if email_group == "legit" else 1
    return out[[
        "text", "label_binary", "source_type", "email_group",
        "sender", "receiver", "date", "subject", "body", "urls", "label"
    ]]

def standardize_llm_df(df, email_group):
    out = df.copy()
    out["text"] = out["text"].fillna("").astype(str)
    out["source_type"] = "llm"
    out["email_group"] = email_group
    out["label_binary"] = 0 if email_group == "legit" else 1
    return out[["text", "label_binary", "source_type", "email_group", "label"]]

In [8]:
# Unificar datasets

human_legit_std = standardize_human_df(human_legit, "legit")
human_phishing_std = standardize_human_df(human_phishing, "phishing")
llm_legit_std = standardize_llm_df(llm_legit, "legit")
llm_phishing_std = standardize_llm_df(llm_phishing, "phishing")

df_all = pd.concat(
    [human_legit_std, human_phishing_std, llm_legit_std, llm_phishing_std],
    ignore_index=True
)

print("Shape del dataset unificado:", df_all.shape)
display(df_all.head())

Shape del dataset unificado: (4000, 11)


,text,label_binary,source_type,email_group,sender,receiver,date,subject,body,urls,label
0,"Starting IC with wizard Hi\r\n\r\n\t\tI am running the IR test program from Max Davies. To do this I need to start\r\nIC with thw wizard option. As I have a macintosh, there is a wizard option for...",0,human,legit,Jesus Miguel Recuenco Ezquerra <JMRECU@teleline.es>,handy board <handyboard@media.mit.edu>,2019-10-29 22:53:50,Starting IC with wizard,"Hi\r\n\r\n\t\tI am running the IR test program from Max Davies. To do this I need to start\r\nIC with thw wizard option. As I have a macintosh, there is a wizard option for\r\nthe mac version of I...",0,0
1,Trade Me -- A question on your auction: Auction 130939357 for Small\tbookshelf \r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\nSecurity Note: Trade Me will never ask you for your password via ...,0,human,legit,Trade Me <xfnbqb@trademe.co.nz>,user2.4@gvc.ceas-challenge.cc,2008-08-06 13:53:26,Trade Me -- A question on your auction: Auction 130939357 for Small\tbookshelf,\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\nSecurity Note: Trade Me will never ask you for your password via email\r\n\r\n\r\n\r\nThis is an automated email regarding listing reference: 13...,0,0
2,"Trade Me - A request from a Trade Me member. Auction: 129227503 \r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\nTrade Me Offer RequestGenerated 8 December, 6:27 AM\r\n\r\n\r\nSecurity Note: Trade Me will ...",0,human,legit,Trade Me <xfnbqb@trademe.co.nz>,user2.4@gvc.ceas-challenge.cc,2008-08-06 13:45:53,Trade Me - A request from a Trade Me member. Auction: 129227503,"\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\n\r\nTrade Me Offer RequestGenerated 8 December, 6:27 AM\r\n\r\n\r\nSecurity Note: Trade Me will never ask you for your password via email\r\n\r\nHi Tony,\r\nA T...",0,0
3,"RE: NorthTec Account/Password Hi Tony\r\nNot sure why it didn't work, but I manually set it to that from this end so you should be sweet to log in with that now.\r\n\r\nRegards,\r\nKevin Jacobson\...",0,human,legit,Kevin Jacobson <wffjeanja@northtec.ac.nz>,user2.1@gvc.ceas-challenge.cc,2008-08-06 13:43:27,RE: NorthTec Account/Password,"Hi Tony\r\nNot sure why it didn't work, but I manually set it to that from this end so you should be sweet to log in with that now.\r\n\r\nRegards,\r\nKevin Jacobson\r\n\r\nICT Services\r\nNorthTe...",1,0
4,2008 timetable Kindly suggest changes\r\n\r\n------------------------------------------------\r\nDr. Albert van Aardt\r\nPrincipal Academic Staff Member\r\nNorthland Polytechnic\r\nWhangarei\r\nNe...,0,human,legit,Albert van Aardt <zfdrqfguo@northtec.ac.nz>,user2.1@gvc.ceas-challenge.cc,2008-08-06 13:26:57,2008 timetable,Kindly suggest changes\r\n\r\n------------------------------------------------\r\nDr. Albert van Aardt\r\nPrincipal Academic Staff Member\r\nNorthland Polytechnic\r\nWhangarei\r\nNew Zealand\r\nTe...,0,0


In [9]:
# Revisar posible inconsistencia entre carpetas y labels originales

print("Cruce entre email_group y label original:")
display(pd.crosstab(df_all["email_group"], df_all["label"], dropna=False))

Cruce entre email_group y label original:


label,0,1
email_group,,
legit,1000,1000
phishing,0,2000


## Nota importante sobre las etiquetas

Durante la inspección inicial se observa que algunos archivos no siguen de forma consistente la misma convención en la columna `label`.  
Por esa razón, para esta implementación se utilizará como etiqueta principal `label_binary`, derivada de la carpeta de origen:

- `legit` -> 0
- `phishing` -> 1

La columna `label` original se conserva solo como referencia.